In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/FYP

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/FYP


In [31]:
import pandas as pd

# Load the CSV files into pandas DataFrames
data = pd.read_csv('fixed_gear.csv')

data['gear_type']=1

In [32]:
print(data.shape[0])

1559137


In [33]:
import pandas as pd
from geopy.distance import geodesic

# Assuming 'data' is the concatenated DataFrame with the gear_type column added
# Sort the data by MMSI and timestamp
data_sorted = data.sort_values(by=['mmsi', 'timestamp'])

# Group the data by MMSI
grouped = data_sorted.groupby('mmsi')

In [34]:
import pandas as pd
from geopy.distance import geodesic

# Sort the data by 'mmsi' and 'timestamp'
data_sorted = data.sort_values(by=['mmsi', 'timestamp'])

# Calculate distances between consecutive points
distances = []
for mmsi, group in data_sorted.groupby('mmsi'):
    latitudes = group['lat'].tolist()
    longitudes = group['lon'].tolist()
    dist_list = [geodesic((latitudes[i], longitudes[i]), (latitudes[i + 1], longitudes[i + 1])).meters for i in range(len(latitudes) - 1)]
    dist_list.append(0)  # Add 0 for the last row of each MMSI
    distances.extend(dist_list)

# Add 'distances' column to the DataFrame
data_sorted['distances'] = distances

# Print the first few rows of the updated DataFrame
print(data_sorted.head())

           mmsi     timestamp  distance_from_shore  distance_from_port  speed  \
0  7.572519e+12  1.347664e+09                  0.0        36054.625000    0.0   
1  7.572519e+12  1.348056e+09                  0.0        36054.625000    0.0   
2  7.572519e+12  1.350409e+09                  0.0        90970.296875    0.0   
3  7.572519e+12  1.350410e+09                  0.0        90970.296875    0.0   
4  7.572519e+12  1.350411e+09                  0.0        90970.296875    0.0   

       course        lat       lon  is_fishing source  gear_type     distances  
0    0.000000  42.798748 -8.944992        -1.0    gfw          1      7.587026  
1    0.000000  42.798717 -8.945075        -1.0    gfw          1  40685.859974  
2  198.199997  43.106419 -9.215466        -1.0    gfw          1      3.335203  
3  186.899994  43.106434 -9.215431        -1.0    gfw          1      0.953299  
4  190.500000  43.106430 -9.215442        -1.0    gfw          1      1.308751  


In [35]:
# Check the unique values in the 'source' feature
unique_sources = data['source'].unique()
print(unique_sources)

from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform the 'source' feature to encode the values
data_sorted['source'] = label_encoder.fit_transform(data_sorted['source'])

['gfw' 'dalhousie_longliner' 'false_positives']


In [36]:
# Pre processing function
def preprocess_data(data_sorted):
    label_encoder = LabelEncoder()
    features = []
    labels = []
    for index, row in data_sorted.iterrows():
        features.append({
            'mmsi': row['mmsi'],
            'lat': row['lat'],
            'lon': row['lon'],
            'course': row['course'],
            'distance_from_shore': row['distance_from_shore'],
            'distance_from_port': row['distance_from_port'],
            'speed': row['speed'],
            'gear_type': row['gear_type'],
            'distances': row['distances'],
        })
        labels.append(1 if row['is_fishing'] > 0 else 0)

    features_df = pd.DataFrame(features)

    return features_df, np.array(labels)

In [37]:
import pandas as pd
def check_data_types(data):
  for col in data.columns:
    print(f"Feature: {col}, Type: {data[col].dtype}")
# Convert numerical features to float64
numerical_features = [col for col in data_sorted.columns]

data_sorted = data_sorted.astype({col: 'float64' for col in numerical_features})

print("Data types after conversion:")
check_data_types(data_sorted.copy())

Data types after conversion:
Feature: mmsi, Type: float64
Feature: timestamp, Type: float64
Feature: distance_from_shore, Type: float64
Feature: distance_from_port, Type: float64
Feature: speed, Type: float64
Feature: course, Type: float64
Feature: lat, Type: float64
Feature: lon, Type: float64
Feature: is_fishing, Type: float64
Feature: source, Type: float64
Feature: gear_type, Type: float64
Feature: distances, Type: float64


In [38]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import numpy as np

# Preprocess data by gear_type
preprocessed_data = []
for gear_type, group_data in data_sorted.groupby('gear_type'):
    features, labels = preprocess_data(group_data)  # Call your existing preprocess_data function
    preprocessed_data.extend([(features_row, labels_row) for features_row, labels_row in zip(features.itertuples(index=False), labels)])

In [39]:
# Convert features back to a DataFrame
features_df = pd.DataFrame(features)

In [40]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split data into training, validation, and testing sets
X_train, X_val, y_train, y_val = train_test_split(features_df, labels, test_size=0.2, random_state=42)

In [41]:
# Combine preprocessed features and labels into DataFrames
features_df = pd.DataFrame([data_point[0] for data_point in preprocessed_data])
labels_df = pd.DataFrame([data_point[1] for data_point in preprocessed_data])

# Train the model (example using an LSTM)
from keras.models import Sequential
from keras.layers import LSTM, Dense

In [42]:
X_train.head()

,mmsi,lat,lon,course,distance_from_shore,distance_from_port,speed,gear_type,distances
514404,1.040818e+14,56.697033,8.218382,76.400002,0.000000,1414.178833,0.0,1.0,0.857567
1141209,2.455585e+14,56.368427,8.120448,273.700012,0.000000,47538.289062,0.0,1.0,0.294677
1485640,2.733520e+14,58.112282,11.366404,298.600006,0.000000,33733.429688,0.0,1.0,1.364907
415658,9.400902e+13,60.841034,-2.023723,1.000000,81022.703125,117410.914062,2.0,1.0,454.681732
1036480,2.259868e+14,55.442505,5.276460,208.100006,311841.687500,353026.750000,1.0,1.0,19.566284


In [43]:
import numpy as np

# Convert DataFrame to NumPy array
X_train_array = X_train.values
X_val_array = X_val.values


# Reshape features for LSTM input
X_train_reshaped = X_train_array.reshape((X_train_array.shape[0], X_train_array.shape[1], 1))
X_val_reshaped = X_val_array.reshape((X_val_array.shape[0], X_val_array.shape[1], 1))

In [44]:
from keras.utils import to_categorical

# One-hot encode labels for multi-class classification
y_train_encoded = to_categorical(y_train)
y_val_encoded = to_categorical(y_val)

In [45]:
# Extract the values from the DataFrame
X_train_values = features_df.values

# Reshape X_train_values to be 3D: (num_samples, 1, num_features)
X_train_reshaped = X_train_values.reshape((X_train_values.shape[0], 1, X_train_values.shape[1]))


In [46]:
features_df.head()

,mmsi,lat,lon,course,distance_from_shore,distance_from_port,speed,gear_type,distances
0,7.572519e+12,42.798748,-8.944992,0.000000,0.0,36054.625000,0.0,1.0,7.587026
1,7.572519e+12,42.798717,-8.945075,0.000000,0.0,36054.625000,0.0,1.0,40685.859974
2,7.572519e+12,43.106419,-9.215466,198.199997,0.0,90970.296875,0.0,1.0,3.335203
3,7.572519e+12,43.106434,-9.215431,186.899994,0.0,90970.296875,0.0,1.0,0.953299
4,7.572519e+12,43.106430,-9.215442,190.500000,0.0,90970.296875,0.0,1.0,1.308751


In [47]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# 1. Handling Missing Values:
imputer = SimpleImputer(strategy='median')  # Replace missing values with median
features_df = imputer.fit_transform(features_df)

# 2. Checking Label Distribution:
num_fishing = (labels > 0).sum()
num_not_fishing = (labels == 0).sum()
num_unsure = (labels == -1).sum()
print("Number of unique MMSIs with fishing labels:")
print(f"  - Fishing: {num_fishing}")
print(f"  - Not Fishing: {num_not_fishing}")
print(f"  - Unsure: {num_unsure}")

# 3. Label Encoding:
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)  # Use LabelEncoder for binary classes

# 4. Feature Normalization:
scaler = MinMaxScaler()
normalized_features = scaler.fit_transform(features_df)  # Normalize numerical features

# 5. Reshape for LSTM:
X_train_reshaped = normalized_features.reshape((X_train_reshaped.shape[0], 1, 9))

Number of unique MMSIs with fishing labels:
  - Fishing: 12068
  - Not Fishing: 1547069
  - Unsure: 0


In [55]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense

# Model architecture definition
# Model architecture definition
model = Sequential()
# Conv1D layer with 64 filters and a kernel size of 3
model.add(Conv1D(64, 3, activation='relu', input_shape=(X_train_values.shape[1], 1)))
# MaxPooling1D layer
model.add(MaxPooling1D(2))
# Flatten layer to flatten the output of the convolutional layer
model.add(Flatten())
model.add(Dense(2, activation='softmax'))

from tensorflow.keras.optimizers import Adam

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-3,
    decay_steps=10000,
    decay_rate=0.9
)
optimizer = Adam(learning_rate=lr_schedule)

# Compile the model
model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

Epoch 1/10
31183/31183 [==============================] - 109s 3ms/step - loss: 13124646912.0000 - accuracy: 0.9856 - val_loss: 2143720704.0000 - val_accuracy: 0.9921
Epoch 2/10
31183/31183 [==============================] - 110s 4ms/step - loss: 141620880.0000 - accuracy: 0.9831 - val_loss: 559.7978 - val_accuracy: 0.9919
Epoch 3/10
31183/31183 [==============================] - 110s 4ms/step - loss: 1359827.1250 - accuracy: 0.9861 - val_loss: 178.8389 - val_accuracy: 0.9919
Epoch 4/10
31183/31183 [==============================] - 111s 4ms/step - loss: 2165185.5000 - accuracy: 0.9854 - val_loss: 195.3251 - val_accuracy: 0.9920
Epoch 5/10
31183/31183 [==============================] - 99s 3ms/step - loss: 18356.7188 - accuracy: 0.9850 - val_loss: 185.7962 - val_accuracy: 0.9921
Epoch 6/10
31183/31183 [==============================] - 94s 3ms/step - loss: 252396.0469 - accuracy: 0.9849 - val_loss: 1184.4119 - val_accuracy: 0.5833
Epoch 7/10
31183/31183 [==============================]

In [56]:
model.save('model1_architecture.json')
model.save_weights('model1_weights.h5')